In [ ]:
import os
import re
import string
import fitz  # PyMuPDF
import docx
import spacy
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from openpyxl import Workbook
from openpyxl.styles import PatternFill
from openpyxl.utils.dataframe import dataframe_to_rows
from docx2pdf import convert

# ------------------------------
# Step 0 – NLTK Resources
# ------------------------------
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("wordnet")

# ------------------------------
# Step 1 – Preprocessing Setup
# ------------------------------
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w.isalpha() and w not in stop_words]
    return tokens

# ------------------------------
# Step 2 – Text Extraction
# ------------------------------
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    return "".join([page.get_text() for page in doc])

# ------------------------------
# Step 3 – Convert DOCX to PDF
# ------------------------------
def convert_docx_to_pdf(docx_path, pdf_output_folder=None):
    if pdf_output_folder is None:
        pdf_output_folder = os.path.dirname(docx_path)
    pdf_path = os.path.join(pdf_output_folder, os.path.splitext(os.path.basename(docx_path))[0] + ".pdf")
    convert(docx_path, pdf_path)
    return pdf_path

# ------------------------------
# Step 4 – Extract Multi-word Skills from JD
# ------------------------------
nlp = spacy.load("en_core_web_sm")

def extract_skills_from_jd(jd_text):
    doc = nlp(jd_text)
    skills = set()

    # Noun chunks for multi-word skills
    for chunk in doc.noun_chunks:
        skill = chunk.text.lower().strip()
        if skill:
            skills.add(skill)

    # Proper nouns and uppercase abbreviations
    for token in doc:
        if token.pos_ in ["PROPN", "NOUN"] or token.text.isupper():
            skills.add(token.text.lower())

    # Capture punctuated tech skills like CI/CD, Node.js, C++
    custom_skills = re.findall(r"\b[a-zA-Z0-9./+-]+(?:/[a-zA-Z0-9./+-]+)*\b", jd_text.lower())
    skills.update(s for s in custom_skills if s)

    return skills

# ------------------------------
# Step 5 – Candidate Details (name, email, phone)
# ------------------------------
def _extract_phone(cv_text: str) -> str:
    candidates = re.findall(r'(\+?\d[\d\s\-()]{8,20})', cv_text)
    for c in candidates:
        if re.search(r'(19|20)\d{2}\s*[-–]\s*(19|20)\d{2}', c):
            continue
        digits = re.sub(r'\D', '', c)
        if 10 <= len(digits) <= 15:
            return re.sub(r'\s+', ' ', c).strip()
    return ""

def extract_candidate_details(cv_text, cv_path):
    email_match = re.search(r"[\w\.-]+@[\w\.-]+\.\w+", cv_text)
    email = email_match.group(0) if email_match else ""
    phone = _extract_phone(cv_text)
    name = ""

    # PDF: largest font size
    if cv_path.lower().endswith(".pdf"):
        try:
            doc = fitz.open(cv_path)
            max_font_size, candidate_name = 0, ""
            for page in doc:
                blocks = page.get_text("dict")["blocks"]
                for block in blocks:
                    if "lines" in block:
                        for line in block["lines"]:
                            for span in line["spans"]:
                                text = span["text"].strip()
                                if not text or len(text.split()) > 6:
                                    continue
                                if span["size"] > max_font_size and text.lower() not in ["resume", "curriculum vitae", "cv"]:
                                    max_font_size = span["size"]
                                    candidate_name = text
                if candidate_name:
                    name = candidate_name
                    break
        except Exception:
            pass

    # Fallbacks
    if not name:
        lines = [line.strip() for line in cv_text.splitlines() if line.strip()]
        for line in lines[:8]:
            if "@" in line or re.search(r"\d", line) or any(x in line.lower() for x in ["resume", "curriculum", "cv", "linkedin", "email", "phone"]):
                continue
            if 1 <= len(line.split()) <= 5:
                name = line
                break
    if not name and email:
        local_part = email.split("@")[0]
        name_guess = re.sub(r"[\d._-]", " ", local_part).title()
        name = " ".join(name_guess.split())
    if not name:
        name = os.path.splitext(os.path.basename(cv_path))[0]

    return name, email, phone

# ------------------------------
# Step 6 – Skill Match + Scoring
# ------------------------------
def extract_skills_from_cv(cv_text, jd_skills):
    cv_text_lower = cv_text.lower()
    matched_skills = {skill for skill in jd_skills if skill in cv_text_lower}
    return matched_skills

def calculate_score(cv_skills, jd_skills, cv_text):
    if not jd_skills:
        return 0, set(), 0, ""
    match_count = len(cv_skills.intersection(jd_skills))
    score = (match_count / len(jd_skills)) * 100
    gap = jd_skills - cv_skills

    text_lower = cv_text.lower()
    degree_keywords = [
        "bachelor", "master", "phd", "b.sc", "m.sc", "msc", "mba",
        "university", "degree", "diploma", "bachelors", "masters"
    ]
    exp_keywords_simple = ["experience", "work history", "present", "consultant", "analyst", "engineer", "developer"]
    exp_years_pattern = r'\b\d{1,2}\+?\s*(years|year|yrs)\b'

    has_degree = any(k in text_lower for k in degree_keywords)
    has_experience = any(k in text_lower for k in exp_keywords_simple) or bool(re.search(exp_years_pattern, text_lower))

    bonus = 0
    reasons = []
    if has_degree:
        bonus += 10
        reasons.append("Degree +10")
    if has_experience:
        bonus += 10
        reasons.append("Experience +10")

    score = min(100, score + bonus)
    return score, gap, bonus, "; ".join(reasons)

# ------------------------------
# Step 7 – Rank CVs (PDF-first approach)
# ------------------------------
def rank_cvs(cv_folder, jd_skills):
    results = []
    for cv_file in os.listdir(cv_folder):
        if not (cv_file.lower().endswith(".pdf") or cv_file.lower().endswith(".docx")):
            continue
        if re.search(r'\b(jd|job|position description)\b', cv_file.lower()):
            continue

        cv_path = os.path.join(cv_folder, cv_file)
        try:
            if cv_file.lower().endswith(".pdf"):
                cv_text = extract_text_from_pdf(cv_path)
            else:
                # Convert DOCX → PDF and then read PDF
                pdf_path = convert_docx_to_pdf(cv_path)
                cv_text = extract_text_from_pdf(pdf_path)
        except Exception as e:
            print(f"Failed to read {cv_file}: {e}")
            continue

        name, email, phone = extract_candidate_details(cv_text, cv_path)
        cv_skills = extract_skills_from_cv(cv_text, jd_skills)
        score, gap, bonus_pts, bonus_reasons = calculate_score(cv_skills, jd_skills, cv_text)

        results.append({
            "Candidate File": cv_file,
            "Name": name,
            "Email": email,
            "Phone": phone,
            "Score": round(score, 2),
            "Bonus Points": bonus_pts,
            "Bonus Reasons": bonus_reasons,
            "Matched Skills": ", ".join(sorted(cv_skills)),
            "Missing Skills": ", ".join(sorted(gap)),
            "JD Skills": ", ".join(sorted(jd_skills))
        })

    return sorted(results, key=lambda x: x["Score"], reverse=True)

# ------------------------------
# Step 8 – Read JD from Text File
# ------------------------------
def read_jd_from_file(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Job description file not found: {file_path}")
    with open(file_path, "r", encoding="utf-8") as f:
        jd_text = f.read()
    return jd_text

# ------------------------------
# Step 9 – Save Excel and CSV
# ------------------------------
def save_excel_with_summary(df, save_folder):
    output_excel = os.path.join(save_folder, "cv_ranking_summary.xlsx")
    output_csv = os.path.join(save_folder, "cv_ranking.csv")

    wb = Workbook()
    ws_data = wb.active
    ws_data.title = "CV Ranking"

    # Write data
    for r in dataframe_to_rows(df, index=False, header=True):
        ws_data.append(r)

    # Highlight empty Name/Email/Phone or Missing Skills
    red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    headers = {cell.value: idx for idx, cell in enumerate(ws_data[1], 1)}

    for col_name in ["Name", "Email", "Phone"]:
        if col_name in headers:
            cidx = headers[col_name]
            for row in range(2, ws_data.max_row + 1):
                cell = ws_data.cell(row=row, column=cidx)
                if not cell.value or str(cell.value).strip() == "":
                    cell.fill = red_fill

    if "Missing Skills" in headers:
        cidx = headers["Missing Skills"]
        for row in range(2, ws_data.max_row + 1):
            cell = ws_data.cell(row=row, column=cidx)
            if cell.value and str(cell.value).strip() != "":
                cell.fill = red_fill

    # Summary sheet
    ws_summary = wb.create_sheet(title="Summary")
    total_candidates = len(df)
    average_score = round(df["Score"].mean(), 2) if not df.empty else 0
    top_candidates = df.sort_values(by="Score", ascending=False).head(3)

    ws_summary.append(["Metric", "Value"])
    ws_summary.append(["Total Candidates", total_candidates])
    ws_summary.append(["Average Score", average_score])
    ws_summary.append([])
    ws_summary.append(["Top 3 Candidates", "Score"])
    for _, row in top_candidates.iterrows():
        ws_summary.append([row["Name"] or row["Candidate File"], row["Score"]])

    wb.save(output_excel)
    df.to_csv(output_csv, index=False)

    print(f"Excel saved with summary and highlights: {output_excel}")
    print(f"CSV saved: {output_csv}")

# ------------------------------
# Step 10 – Main Execution
# ------------------------------
if __name__ == "__main__":
    jd_file = r"filename location\CV\job_description.txt"
    cv_folder = r"filename location\CV"

    jd_text = read_jd_from_file(jd_file)
    jd_skills = extract_skills_from_jd(jd_text)

    results = rank_cvs(cv_folder, jd_skills)
    df = pd.DataFrame(results)

    # Display results
    for r in results:
        print(f"Candidate: {r['Name'] or '[No Name]'} ({r['Candidate File']}) - Score: {r['Score']}% "
              f"(Bonus: {r['Bonus Points']} | {r['Bonus Reasons']})")
        print(f"Email: {r['Email'] or '-'} | Phone: {r['Phone'] or '-'}")
        print(f"Matched Skills: {r['Matched Skills']}")
        print(f"Missing Skills: {r['Missing Skills']}")
        print("-" * 60)

    # Save Excel and CSV
    save_excel_with_summary(df, cv_folder)
